# Séance 5 — Données simulées : connaître la vérité pour comprendre les modèles

Avec des données réelles, nous ne connaissons pas la fonction qui a généré les observations.

Avec des données simulées, nous pouvons imposer :

$
Y=f(X)+\varepsilon
$

et vérifier si les modèles retrouvent cette structure.

Objectifs : comprendre le biais, la variance, la complexité et le surapprentissage.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

rng = np.random.default_rng(42)

## 1. Une relation linéaire connue

Nous créons :

$
Y=3+2X+\varepsilon
$

Les vrais paramètres sont donc β₀=3 et β₁=2.

In [ ]:
n = 200
X = rng.uniform(-3, 3, size=(n,1))
epsilon = rng.normal(0,1,size=n)
y = 3 + 2*X[:,0] + epsilon

plt.figure(figsize=(8,5))
plt.scatter(X[:,0], y)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Relation linéaire simulée")
plt.show()

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lr = LinearRegression()
lr.fit(Xtr, ytr)

print("Intercept estimé :", lr.intercept_)
print("Coefficient estimé :", lr.coef_[0])
print("R² test :", lr.score(Xte, yte))

### Question

Pourquoi les estimations ne sont-elles pas exactement 3 et 2 ?

## 2. Relation non linéaire

Cette fois :

$
Y=X^2+\varepsilon
$

Nous savons donc qu'une droite ne peut pas représenter parfaitement la relation.

In [ ]:
X = rng.uniform(-3,3,size=(200,1))
y = X[:,0]**2 + rng.normal(0,1,size=200)

plt.figure(figsize=(8,5))
plt.scatter(X[:,0], y)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Relation quadratique simulée")
plt.show()

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modeles = {
    "Linéaire": LinearRegression(),
    "KNN": KNeighborsRegressor(n_neighbors=10),
    "Arbre": DecisionTreeRegressor(max_depth=4, random_state=42),
    "Forêt": RandomForestRegressor(
        n_estimators=200, max_depth=5, random_state=42
    )
}

for nom, modele in modeles.items():
    modele.fit(Xtr,ytr)
    pred = modele.predict(Xte)
    print(
        nom,
        "R² =", round(r2_score(yte,pred),3),
        "RMSE =", round(np.sqrt(mean_squared_error(yte,pred)),3)
    )

In [ ]:
xgrid = np.linspace(-3,3,400).reshape(-1,1)

plt.figure(figsize=(10,6))
plt.scatter(Xtr[:,0],ytr,alpha=0.4,label="train")

for nom,modele in modeles.items():
    plt.plot(xgrid[:,0],modele.predict(xgrid),label=nom)

plt.xlabel("X")
plt.ylabel("Y")
plt.title("Comparaison des modèles")
plt.legend()
plt.show()

### Question

Quel modèle semble le mieux adapté à la relation quadratique ? Pourquoi la régression linéaire est-elle limitée ?

## 3. Le rôle du bruit

Le bruit correspond à une composante de Y que X ne permet pas de prédire.

Plus le bruit augmente, plus même le meilleur modèle aura des erreurs importantes.

In [ ]:
X = rng.uniform(-3,3,size=(200,1))
fig, axes = plt.subplots(1,3,figsize=(15,4))

for ax,sigma in zip(axes,[0.1,1,3]):
    y = X[:,0]**2 + rng.normal(0,sigma,size=200)
    ax.scatter(X[:,0],y)
    ax.set_title(f"bruit : sigma={sigma}")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")

plt.tight_layout()
plt.show()

## 4. Observer le surapprentissage

Nous créons :

$
Y=\sin(X)+\varepsilon
$

et faisons varier la profondeur d'un arbre.

In [ ]:
X = rng.uniform(-3,3,size=(250,1))
y = np.sin(X[:,0]) + rng.normal(0,0.2,size=250)

Xtr,Xte,ytr,yte = train_test_split(
    X,y,test_size=0.3,random_state=42
)

profondeurs=[1,2,3,5,10,None]
res=[]

for d in profondeurs:
    arbre=DecisionTreeRegressor(max_depth=d,random_state=42)
    arbre.fit(Xtr,ytr)
    res.append({
        "profondeur":str(d),
        "R2_train":arbre.score(Xtr,ytr),
        "R2_test":arbre.score(Xte,yte)
    })

res=pd.DataFrame(res)
res

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(res["profondeur"],res["R2_train"],marker="o",label="Train")
plt.plot(res["profondeur"],res["R2_test"],marker="o",label="Test")
plt.xlabel("Profondeur")
plt.ylabel("R²")
plt.title("Biais / variance et surapprentissage")
plt.legend()
plt.show()

### Question finale

Repérez une situation où le R² train augmente alors que le R² test cesse de progresser ou diminue.

C'est exactement le type de comportement que la validation croisée cherche à éviter lors du choix de la complexité.

# Bilan

La simulation permet de contrôler la vérité :

- fonction réelle connue ;
- niveau de bruit connu ;
- complexité contrôlée.

Elle permet donc de comprendre expérimentalement :

**modèle trop simple → biais élevé**

**modèle trop complexe → variance élevée / surapprentissage**

**modèle adapté → bonne généralisation**